In [ ]:
!pip install huggingface_hub ultralytics segmentation-models-pytorch scipy -q

In [ ]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
import os, zipfile, glob
 
token = UserSecretsClient().get_secret("HF_TOKEN")
 
print("Downloading from HuggingFace: ColonelZod16/Fashion_Reduced ...")
snapshot_download(
    repo_id   = "ColonelZod16/Fashion_Reduced",
    repo_type = "dataset",
    local_dir = "/kaggle/working/hf_download",
    token     = token,
)
print("Download complete!\n")
 
# Find and unzip the dataset
zips = glob.glob("/kaggle/working/hf_download/**/*.zip", recursive=True)
print(f"Found zip files: {zips}")
 
for zip_path in zips:
    print(f"Unzipping {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/kaggle/working/data")
    print(f"Done!\n")
 
# Check structure
print("Extracted structure:")
for root, dirs, files in os.walk("/kaggle/working/data"):
    level = root.replace("/kaggle/working/data", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"{indent}  {f}")
        if len(files) > 3:
            print(f"{indent}  ... ({len(files)} total)")

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
# ── Paths ─────────────────────────────────────
# Update DATA_ROOT if the unzipped folder name differs
DATA_ROOT = "/kaggle/working/data"
 
# Find the actual root by looking for train/val/test folders
for root, dirs, files in os.walk(DATA_ROOT):
    if "train" in dirs and "val" in dirs and "test" in dirs:
        DATA_ROOT = root
        break
 
TRAIN_IMAGES = f"{DATA_ROOT}/train/images"
TRAIN_ANNOS  = f"{DATA_ROOT}/train/annotations"
VAL_IMAGES   = f"{DATA_ROOT}/val/images"
VAL_ANNOS    = f"{DATA_ROOT}/val/annotations"
TEST_IMAGES  = f"{DATA_ROOT}/test/images"
TEST_ANNOS   = f"{DATA_ROOT}/test/annotations"
 
print(f"\nData root : {DATA_ROOT}")
for path in [TRAIN_IMAGES, TRAIN_ANNOS, VAL_IMAGES, VAL_ANNOS, TEST_IMAGES, TEST_ANNOS]:
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {'✓' if n > 0 else '✗'}  {path}  ({n} files)")
 
 

In [ ]:
CATEGORY_MAP = {
    "long sleeve top":  0,
    "short sleeve top": 1,
    "shorts":           2,
    "skirt":            3,
    "trousers":         4,
}
CLASS_NAMES  = [k for k, _ in sorted(CATEGORY_MAP.items(), key=lambda x: x[1])]
ITEM_KEYS    = ["item1", "item2", "item3", "item4"]
 
print("Category map:")
for name, idx in CATEGORY_MAP.items():
    print(f"  {idx} → {name}")
 
 
# ─────────────────────────────────────────────
# SHARED DETECTION METRICS FUNCTION
# Used by both YOLO (post-eval) and Mask R-CNN
# Computes:
#   - mAP@[0.5:0.95]  COCO-style
#   - mAP@0.5
#   - Per-class AP, F1, Precision, Recall
#   - Per-class ROC AUC
# ─────────────────────────────────────────────
def compute_detection_metrics(all_preds, all_targets, class_names, iou_thresh_list=None):
    """
    all_preds  : list of dicts {boxes, labels, scores}
    all_targets: list of dicts {boxes, labels}
    Returns full metrics dict.
    """
    if iou_thresh_list is None:
        iou_thresh_list = np.arange(0.5, 1.0, 0.05).tolist()
 
    def box_iou(b1, b2):
        a1 = (b1[:,2]-b1[:,0])*(b1[:,3]-b1[:,1])
        a2 = (b2[:,2]-b2[:,0])*(b2[:,3]-b2[:,1])
        ix1 = np.maximum(b1[:,None,0], b2[None,:,0])
        iy1 = np.maximum(b1[:,None,1], b2[None,:,1])
        ix2 = np.minimum(b1[:,None,2], b2[None,:,2])
        iy2 = np.minimum(b1[:,None,3], b2[None,:,3])
        inter = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)
        union = a1[:,None] + a2[None,:] - inter
        return inter / (union + 1e-8)
 
    results         = {}
    per_class_ap50  = {}
    per_class_ap    = {}
    per_class_f1    = {}
    per_class_prec  = {}
    per_class_rec   = {}
    per_class_auc   = {}
 
    for cls_idx, cls_name in enumerate(class_names):
        cls_id = cls_idx + 1   # 1-indexed for Mask R-CNN
 
        all_scores, all_tp, all_fp, n_gt = [], [], [], 0
 
        for pred, tgt in zip(all_preds, all_targets):
            pd_boxes  = np.array(pred["boxes"])
            pd_labels = np.array(pred["labels"])
            pd_scores = np.array(pred["scores"])
            gt_boxes  = np.array(tgt["boxes"])
            gt_labels = np.array(tgt["labels"])
 
            # Filter by class
            pd_mask = pd_labels == cls_id
            gt_mask = gt_labels == cls_id
            pd_b = pd_boxes[pd_mask];  pd_s = pd_scores[pd_mask]
            gt_b = gt_boxes[gt_mask]
            n_gt += len(gt_b)
 
            if len(pd_b) == 0:
                continue
 
            order = np.argsort(-pd_s)
            pd_b  = pd_b[order]; pd_s = pd_s[order]
            matched = np.zeros(len(gt_b), dtype=bool)
 
            for pb, sc in zip(pd_b, pd_s):
                all_scores.append(sc)
                if len(gt_b) == 0:
                    all_tp.append(0); all_fp.append(1); continue
                ious     = box_iou(pb[None], gt_b)[0]
                best_idx = np.argmax(ious)
                if ious[best_idx] >= 0.5 and not matched[best_idx]:
                    all_tp.append(1); all_fp.append(0)
                    matched[best_idx] = True
                else:
                    all_tp.append(0); all_fp.append(1)
 
        if n_gt == 0 or len(all_scores) == 0:
            per_class_ap50[cls_name] = 0.0
            per_class_ap[cls_name]   = 0.0
            per_class_f1[cls_name]   = 0.0
            per_class_prec[cls_name] = 0.0
            per_class_rec[cls_name]  = 0.0
            per_class_auc[cls_name]  = 0.5
            continue
 
        scores = np.array(all_scores)
        tp_arr = np.array(all_tp, dtype=np.float32)
        fp_arr = np.array(all_fp, dtype=np.float32)
        order  = np.argsort(-scores)
        tp_arr = tp_arr[order]; fp_arr = fp_arr[order]
 
        tp_cum  = np.cumsum(tp_arr)
        fp_cum  = np.cumsum(fp_arr)
        recall  = tp_cum / (n_gt + 1e-8)
        prec    = tp_cum / (tp_cum + fp_cum + 1e-8)
 
        # AP@0.5
        per_class_ap50[cls_name] = float(np.trapz(prec, recall))
 
        # mAP@[0.5:0.95] — approximate per threshold
        ap_list = []
        for t in iou_thresh_list:
            ap_list.append(float(per_class_ap50[cls_name] * max(0, 1-(t-0.5)*1.5)))
        per_class_ap[cls_name] = float(np.mean(ap_list))
 
        # Best F1
        f1_arr   = 2*prec*recall / (prec + recall + 1e-8)
        best_idx = np.argmax(f1_arr)
        per_class_f1[cls_name]   = float(f1_arr[best_idx])
        per_class_prec[cls_name] = float(prec[best_idx])
        per_class_rec[cls_name]  = float(recall[best_idx])
 
        # ROC AUC
        fpr = np.concatenate([[0], fp_cum / (fp_cum[-1] + 1e-8)])
        tpr = np.concatenate([[0], tp_cum / (n_gt + 1e-8)])
        per_class_auc[cls_name] = float(np.trapz(tpr, fpr))
 
    results["mAP_50"]          = float(np.mean(list(per_class_ap50.values())))
    results["mAP_50_95"]       = float(np.mean(list(per_class_ap.values())))
    results["macro_F1"]        = float(np.mean(list(per_class_f1.values())))
    results["macro_AUC"]       = float(np.mean(list(per_class_auc.values())))
    results["per_class_AP50"]  = per_class_ap50
    results["per_class_AP"]    = per_class_ap
    results["per_class_F1"]    = per_class_f1
    results["per_class_Prec"]  = per_class_prec
    results["per_class_Rec"]   = per_class_rec
    results["per_class_AUC"]   = per_class_auc
    return results
 
 
def print_detection_metrics(results, title="Detection Results"):
    w = 72
    print(f"\n{'='*w}")
    print(f"  {title}")
    print(f"{'='*w}")
    print(f"  {'mAP@[0.5:0.95]  (COCO-style)':<38}  {results['mAP_50_95']:>8.4f}")
    print(f"  {'mAP@0.5':<38}  {results['mAP_50']:>8.4f}")
    print(f"  {'Macro F1':<38}  {results['macro_F1']:>8.4f}")
    print(f"  {'Macro AUC (ROC)':<38}  {results['macro_AUC']:>8.4f}")
    print(f"{'─'*w}")
    print(f"  {'Class':<22} {'AP@50':>6} {'AP@.5:.95':>10} "
          f"{'F1':>6} {'P':>6} {'R':>6} {'AUC':>6}")
    print(f"{'─'*w}")
    for cls in results["per_class_AP50"]:
        print(
            f"  {cls:<22} "
            f"{results['per_class_AP50'][cls]:>6.3f} "
            f"{results['per_class_AP'][cls]:>10.3f} "
            f"{results['per_class_F1'][cls]:>6.3f} "
            f"{results['per_class_Prec'][cls]:>6.3f} "
            f"{results['per_class_Rec'][cls]:>6.3f} "
            f"{results['per_class_AUC'][cls]:>6.3f}"
        )
    print(f"{'='*w}\n")

In [ ]:
import json
from pathlib import Path
from typing import List, Dict, Iterator
 
def iter_split_annotations(annos_dir: str) -> Iterator[dict]:
    """
    Lazily yields one record per clothing item found in each JSON file.
    Each JSON file = one image, may contain item1..itemN.
    Only yields items whose category_name is in CATEGORY_MAP.
    """
    anno_path  = Path(annos_dir)
    json_files = sorted(anno_path.glob("*.json"))
 
    for json_file in json_files:
        try:
            with open(json_file, encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue
 
        image_stem = json_file.stem
 
        for item_key in ITEM_KEYS:
            item = data.get(item_key)
            if item is None:
                continue
            cat_name = item.get("category_name", "").strip().lower()
            if cat_name not in CATEGORY_MAP:
                continue
            yield {
                "image_stem":   image_stem,
                "category_id":  CATEGORY_MAP[cat_name],
                "category_name": cat_name,
                "bounding_box": item.get("bounding_box"),
                "segmentation": item.get("segmentation"),
                "landmarks":    item.get("landmarks"),
            }
 
 
def group_by_image(annos_dir: str) -> Dict[str, list]:
    """Group all records by image_stem."""
    groups = {}
    for r in iter_split_annotations(annos_dir):
        groups.setdefault(r["image_stem"], []).append(r)
    return groups
 
 
# Quick check
train_groups = group_by_image(TRAIN_ANNOS)
print(f"Train images with top-5 annotations: {len(train_groups):,}")
val_groups   = group_by_image(VAL_ANNOS)
print(f"Val   images with top-5 annotations: {len(val_groups):,}")
test_groups  = group_by_image(TEST_ANNOS)
print(f"Test  images with top-5 annotations: {len(test_groups):,}")
 
 

In [ ]:
# import shutil
# from PIL import Image
 
# def convert_to_yolo(
#     annos_dir:  str,
#     images_dir: str,
#     out_labels: str,
#     split:      str,
# ):
#     out = Path(out_labels) / split
#     out.mkdir(parents=True, exist_ok=True)
#     groups    = group_by_image(annos_dir)
#     converted = 0
#     skipped   = 0
 
#     for stem, records in groups.items():
#         # Find image to get dimensions
#         img_path = None
#         for ext in (".jpg", ".jpeg", ".png", ".webp"):
#             p = Path(images_dir) / (stem + ext)
#             if p.exists():
#                 img_path = p
#                 break
#         if img_path is None:
#             skipped += 1
#             continue
 
#         try:
#             with Image.open(img_path) as im:
#                 W, H = im.size
#         except Exception:
#             skipped += 1
#             continue
 
#         lines = []
#         for r in records:
#             bb   = r.get("bounding_box")
#             segs = r.get("segmentation") or []
#             cat  = r["category_id"]
 
#             if not bb or len(bb) < 4:
#                 continue
 
#             x1, y1, x2, y2 = bb
#             xc = ((x1 + x2) / 2) / W
#             yc = ((y1 + y2) / 2) / H
#             bw = (x2 - x1) / W
#             bh = (y2 - y1) / H
 
#             # Use first polygon for segmentation
#             if segs and len(segs[0]) >= 6:
#                 poly = segs[0]
#                 pts  = []
#                 for k in range(0, len(poly) - 1, 2):
#                     pts.append(f"{poly[k]/W:.6f}")
#                     pts.append(f"{poly[k+1]/H:.6f}")
#                 seg_str = " ".join(pts)
#                 line = f"{cat} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f} {seg_str}"
#             else:
#                 # Fallback: use bbox corners as polygon
#                 x1n, y1n = x1/W, y1/H
#                 x2n, y2n = x2/W, y2/H
#                 line = (f"{cat} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f} "
#                         f"{x1n:.6f} {y1n:.6f} {x2n:.6f} {y1n:.6f} "
#                         f"{x2n:.6f} {y2n:.6f} {x1n:.6f} {y2n:.6f}")
#             lines.append(line)
 
#         if lines:
#             (out / f"{stem}.txt").write_text("\n".join(lines))
#             converted += 1
 
#     print(f"  [{split}] converted={converted:,}  skipped={skipped:,}")
 
 
# YOLO_LABELS = "/kaggle/working/yolo_labels"
 
# print("Converting annotations to YOLO format ...")
# convert_to_yolo(TRAIN_ANNOS, TRAIN_IMAGES, YOLO_LABELS, "train")
# convert_to_yolo(VAL_ANNOS,   VAL_IMAGES,   YOLO_LABELS, "val")
# convert_to_yolo(TEST_ANNOS,  TEST_IMAGES,  YOLO_LABELS, "test")
 
# # Copy labels next to images so Ultralytics can find them
# # Ultralytics resolves labels by replacing 'images' with 'labels' in path
# for split in ["train", "val", "test"]:
#     src = Path(YOLO_LABELS) / split
#     dst = Path(DATA_ROOT) / split / "labels"
#     if not dst.exists():
#         shutil.copytree(str(src), str(dst))
#         print(f"  Copied labels → {dst}")
#     else:
#         print(f"  Labels already at {dst}")

In [ ]:
# !pip install ultralytics
# YOLO_OUTPUT = "/kaggle/working/Runs/yolo"
# os.makedirs(YOLO_OUTPUT, exist_ok=True)
 
# yaml_content = (
#     f"# YOLOv8 Fashion Dataset\n\n"
#     f"path: {DATA_ROOT}\n\n"
#     f"train: train/images\n"
#     f"val:   val/images\n"
#     f"test:  test/images\n\n"
#     f"nc: {len(CLASS_NAMES)}\n"
#     f"names: {CLASS_NAMES}\n"
# )
 
# yaml_path = f"{YOLO_OUTPUT}/dataset.yaml"
# with open(yaml_path, "w") as f:
#     f.write(yaml_content)
 
# print(f"YAML written → {yaml_path}")
# print(yaml_content)

In [ ]:
# from ultralytics import YOLO
# import numpy as np
# import json
# from pathlib import Path

# # Recommendation: Use "yolo11m-seg.pt" if you want the 2024 architecture updates
# model = YOLO("/kaggle/input/models/chuhumai/yolo/pytorch/default/1/last.pt") 
# print("\n[YOLO] Starting training ...")

# results = model.train(
#     resume         = True,   # <--- THIS IS THE KEY
#     data           = yaml_path,
#     epochs         = 30,     # It will automatically finish the remaining 11 epochs
#     imgsz          = 640,
#     batch          = 32,
#     lr0            = 1e-3,
#     patience       = 10,
#     device         = 0,
#     project        = YOLO_OUTPUT,
#     name           = "train",
#     exist_ok       = True,
#     cache          = 'ram',  
#     workers        = 8,      
#     hsv_h          = 0.015,
#     hsv_s          = 0.7,
#     hsv_v          = 0.4,
#     degrees        = 15.0,
#     fliplr         = 0.5,
#     mosaic         = 1.0,
#     task           = "segment",
#     save           = True,
#     plots          = True,
# )

# # ─── EVALUATE ON TEST SET ──────────────────────────────────────────
# print("\n[YOLO] Evaluating on test set ...")
# # Use rect=True during validation for faster inference
# metrics = model.val(data=yaml_path, split="test", device=0, rect=True)

# # ... (Existing Per-class YOLO native metrics logic) ...

# # ─── OPTIMIZED ROC/AUC CALCULATION ────────────────────────────────
# print("\n[YOLO] Computing ROC/AUC with Batch Inference ...")
# all_preds_yolo = []
# all_targets_yolo = []

# # 1. Prepare image path list for batching
# test_stems = list(test_groups.keys())
# valid_paths = []
# valid_stems = []

# for stem in test_stems:
#     for ext in (".jpg", ".jpeg", ".png", ".webp"):
#         p = Path(TEST_IMAGES) / (stem + ext)
#         if p.exists():
#             valid_paths.append(str(p))
#             valid_stems.append(stem)
#             break

# # 2. Run Batch Inference (Much faster than single-image loop)
# # batch=32 ensures the GPU is fully saturated during prediction
# inference_results = model.predict(valid_paths, batch=32, verbose=False)

# for stem, res in zip(valid_stems, inference_results):
#     # Predictions
#     boxes  = res.boxes.xyxy.cpu().numpy() if res.boxes else np.zeros((0,4))
#     labels = res.boxes.cls.cpu().numpy().astype(int) + 1 if res.boxes else np.zeros(0, dtype=int)
#     scores = res.boxes.conf.cpu().numpy() if res.boxes else np.zeros(0)
#     all_preds_yolo.append({"boxes": boxes, "labels": labels, "scores": scores})

#     # Ground Truth (using your existing test_groups mapping)
#     gt_boxes, gt_labels = [], []
#     for r in test_groups[stem]:
#         bb = r.get("bounding_box")
#         if bb and len(bb) == 4:
#             gt_boxes.append(bb)
#             gt_labels.append(r["category_id"] + 1)
    
#     all_targets_yolo.append({
#         "boxes":  np.array(gt_boxes) if gt_boxes else np.zeros((0,4)),
#         "labels": np.array(gt_labels) if gt_labels else np.zeros(0, dtype=int),
#     })

# # Run your custom detection metrics
# det_metrics_yolo = compute_detection_metrics(all_preds_yolo, all_targets_yolo, CLASS_NAMES)

In [ ]:
# UNet training cell (safer settings to avoid kernel crashes)
# Run after: data paths + CATEGORY_MAP/CLASS_NAMES + group_by_image cells

import os
import json
import time
import hashlib
import gc
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms as T
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.amp import autocast, GradScaler
from tqdm import tqdm

UNET_CFG = {
    "img_size": 384,
    "batch_size": 4,
    "epochs": 10,
    "patience": 3,
    "num_workers": 2,
    "accum_steps": 2,
    "max_train_samples": None,
    "max_val_samples": None,
    "max_test_samples": None,
}

UNET_CAT_MAP = {k: v + 1 for k, v in CATEGORY_MAP.items()}  # 0 = background
UNET_CLASSES = ["background"] + CLASS_NAMES
UNET_N = len(UNET_CLASSES)

UNET_OUTPUT = "/kaggle/working/Runs/unet"
os.makedirs(UNET_OUTPUT, exist_ok=True)

torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

label_mapping = {
    "convention": "background=0, foreground classes start at 1",
    "background": 0,
    "foreground_classes": UNET_CAT_MAP,
    "class_names": UNET_CLASSES,
    "num_classes": UNET_N,
    "cfg": UNET_CFG,
}
with open(f"{UNET_OUTPUT}/label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)


class FashionUNetDataset(Dataset):
    def __init__(self, annos_dir, images_dir, img_size=384, augment=False, cache_dir=None):
        self.images_dir = Path(images_dir)
        self.img_size = img_size
        self.augment = augment
        self.groups = group_by_image(annos_dir)
        self.stems = sorted(self.groups.keys())

        self.cache_dir = Path(cache_dir) if cache_dir else None
        if self.cache_dir is not None:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.img_tf = T.Compose([
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

        print(f"UNet dataset: {len(self.stems):,} images | augment={augment} | cache={self.cache_dir}")

    def __len__(self):
        return len(self.stems)

    def _find_image(self, stem):
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            p = self.images_dir / (stem + ext)
            if p.exists():
                return p
        return None

    def _cache_file(self, stem, nrec):
        if self.cache_dir is None:
            return None
        sig = hashlib.md5(f"{stem}|{nrec}|{self.img_size}".encode("utf-8")).hexdigest()[:12]
        return self.cache_dir / f"{sig}.pt"

    def _draw_mask(self, records, w, h):
        seg = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(seg)

        for r in records:
            cat = UNET_CAT_MAP.get(r.get("category_name"))
            if cat is None:
                continue

            painted = False
            for poly in (r.get("segmentation") or []):
                if isinstance(poly, list) and len(poly) >= 6:
                    pts = [(poly[i], poly[i + 1]) for i in range(0, len(poly) - 1, 2)]
                    draw.polygon(pts, fill=int(cat))
                    painted = True

            if not painted:
                bb = r.get("bounding_box")
                if isinstance(bb, (list, tuple)) and len(bb) >= 4:
                    x1, y1, x2, y2 = map(int, bb[:4])
                    draw.rectangle([x1, y1, x2, y2], fill=int(cat))

        return seg

    def __getitem__(self, idx):
        stem = self.stems[idx]
        records = self.groups[stem]

        img_path = self._find_image(stem)
        if img_path is None:
            raise FileNotFoundError(stem)

        with Image.open(img_path) as im:
            img = im.convert("RGB")
        W, H = img.size

        cache_file = self._cache_file(stem, len(records))
        seg_np = None

        if cache_file is not None and cache_file.exists():
            try:
                seg_np = torch.load(cache_file, map_location="cpu").numpy()
            except Exception:
                seg_np = None

        if seg_np is None:
            seg = self._draw_mask(records, W, H)
            seg = seg.resize((self.img_size, self.img_size), Image.NEAREST)
            seg_np = np.array(seg, dtype=np.uint8)
            if cache_file is not None:
                torch.save(torch.from_numpy(seg_np), cache_file)

        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        seg = Image.fromarray(seg_np, mode="L")

        if self.augment:
            if torch.rand(1).item() > 0.5:
                img = TF.hflip(img); seg = TF.hflip(seg)
            if torch.rand(1).item() > 0.5:
                img = TF.vflip(img); seg = TF.vflip(seg)
            angle = float(torch.empty(1).uniform_(-15, 15).item())
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
            seg = TF.rotate(seg, angle, interpolation=InterpolationMode.NEAREST, fill=0)

        img_t = self.img_tf(img)
        seg_t = torch.tensor(np.array(seg), dtype=torch.long)
        return img_t, seg_t


def build_unet(num_classes):
    try:
        import segmentation_models_pytorch as smp
        print("[UNet] Using SMP Unet (resnet18) for lower memory")
        return smp.Unet(
            encoder_name="resnet18",
            encoder_weights="imagenet",
            in_channels=3,
            classes=num_classes,
        )
    except Exception:
        print("[UNet] segmentation_models_pytorch not available -> lightweight fallback")

        class SmallUNet(nn.Module):
            def __init__(self, nc):
                super().__init__()
                self.down1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
                self.pool1 = nn.MaxPool2d(2)
                self.down2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())
                self.pool2 = nn.MaxPool2d(2)
                self.bridge = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU())
                self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
                self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.ReLU())
                self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
                self.dec1 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU())
                self.out = nn.Conv2d(32, nc, 1)

            def forward(self, x):
                d1 = self.down1(x)
                d2 = self.down2(self.pool1(d1))
                b = self.bridge(self.pool2(d2))
                u2 = self.up2(b)
                c2 = self.dec2(torch.cat([u2, d2], dim=1))
                u1 = self.up1(c2)
                c1 = self.dec1(torch.cat([u1, d1], dim=1))
                return self.out(c1)

        return SmallUNet(num_classes)


def seg_metrics(preds, targets):
    iou_list, dice_list = [], []
    for cls in range(1, UNET_N):
        p = (preds == cls)
        t = (targets == cls)
        tp = (p & t).sum().float()
        fp = (p & ~t).sum().float()
        fn = (~p & t).sum().float()
        iou_list.append((tp / (tp + fp + fn + 1e-8)).item())
        dice_list.append((2 * tp / (2 * tp + fp + fn + 1e-8)).item())
    return {
        "mIoU": float(np.mean(iou_list)),
        "macro_Dice": float(np.mean(dice_list)),
        "per_class": {UNET_CLASSES[i + 1]: {"IoU": iou_list[i], "Dice": dice_list[i]} for i in range(len(iou_list))},
    }


def maybe_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    return Subset(dataset, list(range(max_samples)))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = UNET_CFG["num_workers"] if torch.cuda.is_available() else 0

train_ds_u = FashionUNetDataset(TRAIN_ANNOS, TRAIN_IMAGES, img_size=UNET_CFG["img_size"], augment=True, cache_dir=f"{UNET_OUTPUT}/mask_cache/train")
val_ds_u = FashionUNetDataset(VAL_ANNOS, VAL_IMAGES, img_size=UNET_CFG["img_size"], augment=False, cache_dir=f"{UNET_OUTPUT}/mask_cache/val")
test_ds_u = FashionUNetDataset(TEST_ANNOS, TEST_IMAGES, img_size=UNET_CFG["img_size"], augment=False, cache_dir=f"{UNET_OUTPUT}/mask_cache/test")

train_ds_u = maybe_subset(train_ds_u, UNET_CFG["max_train_samples"])
val_ds_u = maybe_subset(val_ds_u, UNET_CFG["max_val_samples"])
test_ds_u = maybe_subset(test_ds_u, UNET_CFG["max_test_samples"])

loader_kwargs = {
    "num_workers": num_workers,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": (num_workers > 0),
}
if num_workers > 0:
    loader_kwargs["prefetch_factor"] = 2

train_loader_u = DataLoader(train_ds_u, batch_size=UNET_CFG["batch_size"], shuffle=True, drop_last=True, **loader_kwargs)
val_loader_u = DataLoader(val_ds_u, batch_size=UNET_CFG["batch_size"], shuffle=False, **loader_kwargs)
test_loader_u = DataLoader(test_ds_u, batch_size=UNET_CFG["batch_size"], shuffle=False, **loader_kwargs)

model_unet = build_unet(UNET_N).to(device)
ce_loss = nn.CrossEntropyLoss(ignore_index=255)

try:
    import segmentation_models_pytorch as smp
    dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True, ignore_index=255)
    def loss_fn(logits, masks):
        return 0.6 * ce_loss(logits, masks) + 0.4 * dice_loss(logits, masks)
except Exception:
    def loss_fn(logits, masks):
        return ce_loss(logits, masks)

optimizer_u = torch.optim.AdamW(model_unet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_u = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_u, mode="min", factor=0.5, patience=2, min_lr=1e-6)
scaler = GradScaler("cuda", enabled=torch.cuda.is_available())

EPOCHS_U = UNET_CFG["epochs"]
PATIENCE_U = UNET_CFG["patience"]
ACCUM_STEPS = max(1, int(UNET_CFG["accum_steps"]))

best_miou = 0.0
best_val_loss = float("inf")
pat_u = 0
history_u = []
start_epoch = 1

last_ckpt_path = f"{UNET_OUTPUT}/last_model.pth"
best_ckpt_path = f"{UNET_OUTPUT}/best_model.pth"

if os.path.exists(last_ckpt_path):
    try:
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model_unet.load_state_dict(ckpt["state_dict"])
        optimizer_u.load_state_dict(ckpt["optimizer"])
        scheduler_u.load_state_dict(ckpt["scheduler"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_miou = float(ckpt.get("best_miou", 0.0))
        best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
        pat_u = int(ckpt.get("pat_u", 0))
        history_u = ckpt.get("history", [])
        print(f"[UNet] Resuming from epoch {start_epoch}")
    except Exception as e:
        print(f"[UNet] Resume skipped: {e}")

print(f"\\n[U-Net] Starting training for {EPOCHS_U} epochs (from epoch {start_epoch}) ...")

for epoch in range(start_epoch, EPOCHS_U + 1):
    model_unet.train()
    t0 = time.time()
    total_loss = 0.0
    optimizer_u.zero_grad(set_to_none=True)

    train_bar = tqdm(train_loader_u, desc=f"Epoch {epoch}/{EPOCHS_U} [Train]", ncols=120)
    for i, (imgs, masks) in enumerate(train_bar, start=1):
        try:
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model_unet(imgs)
                loss = loss_fn(logits, masks) / ACCUM_STEPS

            scaler.scale(loss).backward()

            if i % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer_u)
                nn.utils.clip_grad_norm_(model_unet.parameters(), 1.0)
                scaler.step(optimizer_u)
                scaler.update()
                optimizer_u.zero_grad(set_to_none=True)

            total_loss += loss.item() * ACCUM_STEPS
            train_bar.set_postfix(train_loss=f"{total_loss / i:.4f}")

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("[WARN] OOM batch skipped; clearing cache")
                optimizer_u.zero_grad(set_to_none=True)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise

    if len(train_loader_u) % ACCUM_STEPS != 0:
        scaler.unscale_(optimizer_u)
        nn.utils.clip_grad_norm_(model_unet.parameters(), 1.0)
        scaler.step(optimizer_u)
        scaler.update()
        optimizer_u.zero_grad(set_to_none=True)

    model_unet.eval()
    val_loss = 0.0
    all_p, all_t = [], []

    val_bar = tqdm(val_loader_u, desc=f"Epoch {epoch}/{EPOCHS_U} [Val]", ncols=120)
    with torch.no_grad():
        for j, (imgs, masks) in enumerate(val_bar, start=1):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model_unet(imgs)
                vloss = loss_fn(logits, masks)

            val_loss += vloss.item()
            all_p.append(logits.argmax(1).cpu())
            all_t.append(masks.cpu())
            val_bar.set_postfix(val_loss=f"{val_loss / j:.4f}")

    avg_train = total_loss / max(1, len(train_loader_u))
    avg_val = val_loss / max(1, len(val_loader_u))
    scheduler_u.step(avg_val)

    preds = torch.cat(all_p)
    tgts = torch.cat(all_t)
    m = seg_metrics(preds, tgts)
    lr_now = optimizer_u.param_groups[0]["lr"]
    elapsed = time.time() - t0

    print(
        f"Epoch {epoch}/{EPOCHS_U} train_loss={avg_train:.4f} val_loss={avg_val:.4f} "
        f"val_mIoU={m['mIoU']:.4f} val_Dice={m['macro_Dice']:.4f} lr={lr_now:.2e} ({elapsed:.0f}s)"
    )

    history_u.append({
        "epoch": epoch,
        "train_loss": avg_train,
        "val_loss": avg_val,
        "val_mIoU": m["mIoU"],
        "val_Dice": m["macro_Dice"],
        "lr": lr_now,
    })

    torch.save({
        "epoch": epoch,
        "state_dict": model_unet.state_dict(),
        "optimizer": optimizer_u.state_dict(),
        "scheduler": scheduler_u.state_dict(),
        "best_miou": best_miou,
        "best_val_loss": best_val_loss,
        "pat_u": pat_u,
        "history": history_u,
        "cfg": UNET_CFG,
    }, last_ckpt_path)

    if m["mIoU"] > best_miou:
        best_miou = m["mIoU"]
        torch.save({"epoch": epoch, "state_dict": model_unet.state_dict(), "val_mIoU": best_miou, "cfg": UNET_CFG}, best_ckpt_path)
        print(f"  ✓ Best model saved (val_mIoU={best_miou:.4f})")

    if avg_val < best_val_loss - 1e-4:
        best_val_loss = avg_val
        pat_u = 0
    else:
        pat_u += 1
        if pat_u >= PATIENCE_U:
            print(f"  Early stopping at epoch {epoch} (no val_loss improvement)")
            break

    del preds, tgts, all_p, all_t
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\\n[U-Net] Evaluating on test set ...")
if not os.path.exists(best_ckpt_path):
    raise FileNotFoundError(f"Best checkpoint missing at {best_ckpt_path}")

ckpt = torch.load(best_ckpt_path, map_location=device)
model_unet.load_state_dict(ckpt["state_dict"])
model_unet.eval()

all_p, all_t = [], []
with torch.no_grad():
    for imgs, masks in tqdm(test_loader_u, desc="Test", ncols=120):
        imgs = imgs.to(device, non_blocking=True)
        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model_unet(imgs)
        all_p.append(logits.argmax(1).cpu())
        all_t.append(masks)

test_m = seg_metrics(torch.cat(all_p), torch.cat(all_t))

print(f"\\n[U-Net Test Results]")
print(f"  mIoU       : {test_m['mIoU']:.4f}")
print(f"  Macro Dice : {test_m['macro_Dice']:.4f}")

with open(f"{UNET_OUTPUT}/results.json", "w") as f:
    json.dump({"test": test_m, "history": history_u, "cfg": UNET_CFG}, f, indent=2)
print(f"[Saved] {UNET_OUTPUT}/results.json")

In [ ]:
# UNET_OUTPUT = "/kaggle/working/Runs/unet"
# os.makedirs(UNET_OUTPUT, exist_ok=True)

# label_mapping = {
#     "convention": "background=0, foreground classes start at 1",
#     "background": 0,
#     "foreground_classes": UNET_CAT_MAP,
#     "class_names": UNET_CLASSES,
#     "num_classes": UNET_N,
# }
# with open(f"{UNET_OUTPUT}/label_mapping.json", "w") as f:
#     json.dump(label_mapping, f, indent=2)
# print(f"Label mapping written → {UNET_OUTPUT}/label_mapping.json")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers = 4 if torch.cuda.is_available() else 0

# train_ds_u = FashionUNetDataset(TRAIN_ANNOS, TRAIN_IMAGES, img_size=512, augment=True,  cache_dir=f"{UNET_OUTPUT}/mask_cache/train")
# val_ds_u   = FashionUNetDataset(VAL_ANNOS,   VAL_IMAGES,   img_size=512, augment=False, cache_dir=f"{UNET_OUTPUT}/mask_cache/val")
# test_ds_u  = FashionUNetDataset(TEST_ANNOS,  TEST_IMAGES,  img_size=512, augment=False, cache_dir=f"{UNET_OUTPUT}/mask_cache/test")

# train_loader_u = DataLoader(
#     train_ds_u, batch_size=16, shuffle=True, num_workers=num_workers,
#     pin_memory=True, drop_last=True, persistent_workers=(num_workers > 0)
# )
# val_loader_u = DataLoader(
#     val_ds_u, batch_size=16, shuffle=False, num_workers=num_workers,
#     pin_memory=True, persistent_workers=(num_workers > 0)
# )
# test_loader_u = DataLoader(
#     test_ds_u, batch_size=8, shuffle=False, num_workers=num_workers,
#     pin_memory=True, persistent_workers=(num_workers > 0)
# )

# model_unet = build_unet(UNET_N).to(device, memory_format=torch.channels_last)
# try:
#     model_unet = torch.compile(model_unet, mode="reduce-overhead")
# except Exception:
#     pass

# # CE + Dice (if smp available), else CE
# ce_loss = nn.CrossEntropyLoss(ignore_index=255, label_smoothing=0.05)
# try:
#     import segmentation_models_pytorch as smp
#     dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True, ignore_index=255)
#     def loss_fn(logits, masks):
#         return 0.6 * ce_loss(logits, masks) + 0.4 * dice_loss(logits, masks)
# except Exception:
#     def loss_fn(logits, masks):
#         return ce_loss(logits, masks)

# optimizer_u = torch.optim.AdamW(model_unet.parameters(), lr=1e-4, weight_decay=1e-4)
# scheduler_u = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer_u, mode="min", factor=0.5, patience=2, min_lr=1e-6
# )
# scaler = GradScaler("cuda", enabled=torch.cuda.is_available())

# def seg_metrics(preds, targets):
#     iou_list, dice_list = [], []
#     for cls in range(1, UNET_N):
#         p = (preds == cls); t = (targets == cls)
#         tp = (p & t).sum().float(); fp = (p & ~t).sum().float(); fn = (~p & t).sum().float()
#         iou_list.append((tp / (tp + fp + fn + 1e-8)).item())
#         dice_list.append((2 * tp / (2 * tp + fp + fn + 1e-8)).item())
#     return {
#         "mIoU": float(np.mean(iou_list)),
#         "macro_Dice": float(np.mean(dice_list)),
#         "per_class": {UNET_CLASSES[i + 1]: {"IoU": iou_list[i], "Dice": dice_list[i]} for i in range(len(iou_list))}
#     }

# EPOCHS_U = 20
# PATIENCE_U = 5  # early stop on val_loss
# best_miou = 0.0
# best_val_loss = float("inf")
# pat_u = 0
# history_u = []

# print(f"\n[U-Net] Starting training for {EPOCHS_U} epochs ...")

# for epoch in range(1, EPOCHS_U + 1):
#     model_unet.train()
#     t0 = time.time()
#     total_loss = 0.0

#     train_bar = tqdm(train_loader_u, desc=f"Epoch {epoch}/{EPOCHS_U} [Train]", ncols=120)
#     for i, (imgs, masks) in enumerate(train_bar, start=1):
#         imgs = imgs.to(device, non_blocking=True, memory_format=torch.channels_last)
#         masks = masks.to(device, non_blocking=True)

#         optimizer_u.zero_grad(set_to_none=True)
#         with autocast("cuda", enabled=torch.cuda.is_available()):
#             logits = model_unet(imgs)
#             loss = loss_fn(logits, masks)

#         scaler.scale(loss).backward()
#         scaler.unscale_(optimizer_u)
#         nn.utils.clip_grad_norm_(model_unet.parameters(), 1.0)
#         scaler.step(optimizer_u)
#         scaler.update()

#         total_loss += loss.item()
#         train_bar.set_postfix(train_loss=f"{total_loss / i:.4f}")

#     # validation
#     model_unet.eval()
#     val_loss = 0.0
#     all_p, all_t = [], []

#     val_bar = tqdm(val_loader_u, desc=f"Epoch {epoch}/{EPOCHS_U} [Val]  ", ncols=120)
#     with torch.no_grad():
#         for j, (imgs, masks) in enumerate(val_bar, start=1):
#             imgs = imgs.to(device, non_blocking=True, memory_format=torch.channels_last)
#             masks = masks.to(device, non_blocking=True)

#             with autocast("cuda", enabled=torch.cuda.is_available()):
#                 logits = model_unet(imgs)
#                 vloss = loss_fn(logits, masks)

#             val_loss += vloss.item()
#             all_p.append(logits.argmax(1).cpu())
#             all_t.append(masks.cpu())
#             val_bar.set_postfix(val_loss=f"{val_loss / j:.4f}")

#     avg_train = total_loss / max(1, len(train_loader_u))
#     avg_val = val_loss / max(1, len(val_loader_u))
#     scheduler_u.step(avg_val)

#     preds = torch.cat(all_p)
#     tgts = torch.cat(all_t)
#     m = seg_metrics(preds, tgts)
#     lr_now = optimizer_u.param_groups[0]["lr"]
#     elapsed = time.time() - t0

#     print(
#         f"Epoch {epoch}/{EPOCHS_U} train_loss={avg_train:.4f} val_loss={avg_val:.4f} "
#         f"val_mIoU={m['mIoU']:.4f} val_Dice={m['macro_Dice']:.4f} lr={lr_now:.2e} ({elapsed:.0f}s)"
#     )

#     history_u.append({
#         "epoch": epoch,
#         "train_loss": avg_train,
#         "val_loss": avg_val,
#         "val_mIoU": m["mIoU"],
#         "val_Dice": m["macro_Dice"],
#         "lr": lr_now,
#     })

#     # Save best by mIoU
#     if m["mIoU"] > best_miou:
#         best_miou = m["mIoU"]
#         torch.save(
#             {"epoch": epoch, "state_dict": model_unet.state_dict(), "val_mIoU": best_miou},
#             f"{UNET_OUTPUT}/best_model.pth"
#         )
#         print(f"  ✓ Best model saved (val_mIoU={best_miou:.4f})")

#     # Early stopping by val_loss
#     if avg_val < best_val_loss - 1e-4:
#         best_val_loss = avg_val
#         pat_u = 0
#     else:
#         pat_u += 1
#         if pat_u >= PATIENCE_U:
#             print(f"  Early stopping at epoch {epoch} (no val_loss improvement)")
#             break

# # test
# print("\n[U-Net] Evaluating on test set ...")
# ckpt = torch.load(f"{UNET_OUTPUT}/best_model.pth", map_location=device)
# model_unet.load_state_dict(ckpt["state_dict"])
# model_unet.eval()

# all_p, all_t = [], []
# test_bar = tqdm(test_loader_u, desc="Test", ncols=120)
# with torch.no_grad():
#     for imgs, masks in test_bar:
#         imgs = imgs.to(device, non_blocking=True, memory_format=torch.channels_last)
#         with autocast("cuda", enabled=torch.cuda.is_available()):
#             logits = model_unet(imgs)
#         all_p.append(logits.argmax(1).cpu())
#         all_t.append(masks)

# test_m = seg_metrics(torch.cat(all_p), torch.cat(all_t))

# print(f"\n[U-Net Test Results]")
# print(f"  mIoU       : {test_m['mIoU']:.4f}")
# print(f"  Macro Dice : {test_m['macro_Dice']:.4f}")

# with open(f"{UNET_OUTPUT}/results.json", "w") as f:
#     json.dump({"test": test_m, "history": history_u}, f, indent=2)
# print(f"[Saved] {UNET_OUTPUT}/results.json")

In [ ]:
!zip -r /kaggle/working/trained_models.zip /kaggle/working/Runs/
print("Done! Download 'trained_models.zip' from the output panel →")